In [245]:
import numpy as np

In [246]:
UP, RIGHT, DOWN, LEFT = 0, 1, 2, 3
DIRS = {UP: (-1, 0), RIGHT: (0, 1), DOWN: (1, 0), LEFT: (0, -1)}
ARROWS = {UP: "↑", RIGHT: "→", DOWN: "↓", LEFT: "←"}

In [247]:
class GridWorld:

    def __init__(
        self,
        rows: int,
        cols: int,
        step_reward: float,
        terminals: dict[tuple[int, int], float],
        walls: set[tuple[int, int]],
    ):
        self.rows = rows
        self.cols = cols
        self.step_reward = step_reward
        self.terminals = terminals
        self.walls = walls
        self.s2c = [
            (r, c)
            for r in range(self.rows)
            for c in range(self.cols)
            if (r, c) not in self.walls
        ]
        self.nS = len(self.s2c)
        self.nA = 4
        self.c2s = {cell: i for i, cell in enumerate(self.s2c)}

    def step(self, s: int, a: int):
        dr, dc = DIRS[a]
        cell = self.s2c[s]
        nxt = (cell[0] + dr, cell[1] + dc)
        if (
            not 0 <= nxt[0] < self.rows
            or not 0 <= nxt[1] < self.cols
            or nxt in self.walls
        ):
            nxt = cell
        reward = self.terminals.get(nxt, self.step_reward)
        return self.c2s[nxt], reward
    
    def cell_repr(self, r, c):
        cell = (r, c)
        if cell in self.terminals:
            return f'{self.terminals[cell]:+}'
        elif cell in self.walls:
            return '#'
        else:
            return '·'
        
    def render(self):
        for r in range(self.rows):
            for c in range(self.cols):
                if r == 0 and c == 0:
                    print(" r/c", end="")
                    print(''.join([f'{v:>3} ' for v in range(self.cols)]))
                if c == 0:
                    print(f'{r:>3} ', end="")
                print(f'{self.cell_repr(r, c):>3} ', end="")
            print()
            
    def __repr__(self):
        return f'''
Grid(
    rows={self.rows},
    cols={self.cols},
    step_reward={self.step_reward},
    terminals={self.terminals},
    walls={self.walls},
)
'''

In [248]:
env = GridWorld(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
)

In [249]:
env


Grid(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
)

In [250]:
env.render()

 r/c  0   1   2   3 
  0   ·   ·   ·  +1 
  1   ·   #   ·  -1 
  2   ·   ·   ·   · 


### value iteration

In [251]:
def read_policy(
    env: GridWorld,
    V: np.ndarray,
    gamma=0.9,
):
    policy = np.zeros((env.nS, env.nA))
    for s in range(env.nS):
        if env.s2c[s] in env.terminals:
            continue
        best_a = int(np.argmax(q_from_v(env, V, s, gamma)))
        policy[s, best_a] = 1.0
    return policy

def render_policy(env: GridWorld, policy: np.ndarray):
    for r in range(env.rows):
        for c in range(env.cols):
            if r == 0 and c == 0:
                print(" r/c", end="")
                print(''.join([f'{v:>3} ' for v in range(env.cols)]))
            if c == 0:
                print(f'{r:>3} ', end="")  

            cell = (r, c)
            if cell in env.c2s:
                if cell in env.terminals:
                    value = f'{env.terminals[cell]:+}'
                else:
                    value = ARROWS[int(np.argmax(policy[env.c2s[(r, c)]]))]
            else:
                value = '#'
            print(f'{value:>3} ', end='')
        print()    

def show_V(env: GridWorld, V: np.ndarray):
    for r in range(env.rows):
        for c in range(env.cols):
            if r == 0 and c == 0:
                print("  r/c", end="")
                print(''.join([f'{v:>4} ' for v in range(env.cols)]))
            if c == 0:
                print(f'{r:>4} ', end="")            
            cell = (r, c)
            if cell in env.c2s:
                value = round(V[env.c2s[(r, c)]], 2)
            else:
                value = '#'
            print(f'{value:>4} ', end="")
        print()

def q_from_v(
    env: GridWorld,
    V: np.ndarray,
    s: int,
    gamma: float,
):
    q = np.zeros(env.nA)
    if env.s2c[s] in env.terminals:
        return q
    for a in range(env.nA):
        ns, r = env.step(s, a)
        q[a] = r + gamma * V[ns]
    return q

def value_iteration(
    env: GridWorld,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    verbose=0,
):
    V = np.zeros(env.nS)
    policy = np.zeros((env.nS, env.nA))
    if verbose >= 2:
        print('---------- value_iteration ------------')
        print('V init')
        show_V(env, V)
        print('-' * 25)    
    delta = float('inf')
    i = 0
    while delta >= theta and i < max_iters:
        delta = 0.0
        V_old = V.copy()
        for s in range(env.nS):
            if env.s2c[s] in env.terminals:
                continue
            q = q_from_v(env, V_old, s, gamma)
            V[s] = np.max(q)
            best_a = int(np.argmax(q))
            policy[s] = np.zeros(env.nA)
            policy[s, best_a] = 1.0
            delta = max(delta, abs(V[s] - V_old[s]))
        if verbose >= 1:
            print(f"iter {i}: delta={delta:.6f}")
        if verbose >= 2:
            show_V(env, V)
            print('-' * 25)
        i += 1
    if verbose >= 2:
        render_policy(env, policy)
        print('-' * 25)
    converged = delta < theta
    if verbose >= 1:
        if converged:
                print(f'value_iteration converged in {i - 1} iterations')
        else:
            print(f"value_iteration did not converge in {max_iters} iterations")
    return policy, V, converged

In [252]:
policy, V, converged = value_iteration(env, verbose=2)

---------- value_iteration ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 1: delta=0.900000
  r/c   0    1    2    3 
   0  0.0  0.9  1.0  0.0 
   1  0.0    #  0.9  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 2: delta=0.810000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1  0.0    #  0.9  0.0 
   2  0.0  0.0 0.81  0.0 
-------------------------
iter 3: delta=0.729000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  0.0 
   2  0.0 0.73 0.81 0.73 
-------------------------
iter 4: delta=0.656100
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  0.0 
   2 0.66 0.73 0.81 0.73 
-------------------------
iter 5: delta=0.000000
  r/c   0    1    2    3 
   0 0.81

In [253]:
policy = read_policy(env, V)
render_policy(env, policy)

 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 


### policy iteration

In [254]:
def policy_evaluation(
    env: GridWorld,
    policy: np.ndarray,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    verbose=0,
):
    V = np.zeros(env.nS)
    if verbose >= 2:
        print('---------- policy_evaluation ------------')
        print('V init')
        show_V(env, V)
        print('-' * 25)    
    delta = float('inf')
    i = 0
    while delta >= theta and i < max_iters:
        delta = 0.0
        V_old = V.copy()
        for s in range(env.nS):
            if env.s2c[s] in env.terminals:
                continue
            value = 0.0
            for a in range(env.nA):
                pa = policy[s][a]
                ns, r = env.step(s, a)
                value += pa * (r + gamma * V_old[ns])
            V[s] = value
            delta = max(delta, abs(V[s] - V_old[s]))
        if verbose >= 1:
            print(f"iter {i}: delta={delta:.6f}")
        if verbose >= 2:
            show_V(env, V)
            print('-' * 25)
        i += 1
    converged = delta < theta
    if verbose >= 1:
        if converged:
            print(f'policy_evaluation converged in {i - 1} iterations')
        else:
            print(f"policy_evaluation did not converge in {max_iters} iterations")
    return V, converged

In [255]:
policy, *_ = value_iteration(env)
V, converged = policy_evaluation(env, policy, verbose=2)

---------- policy_evaluation ------------
V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 1: delta=0.900000
  r/c   0    1    2    3 
   0  0.0  0.9  1.0  0.0 
   1  0.0    #  0.9  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 2: delta=0.810000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1  0.0    #  0.9  0.0 
   2  0.0  0.0 0.81  0.0 
-------------------------
iter 3: delta=0.729000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  0.0 
   2  0.0 0.73 0.81 0.73 
-------------------------
iter 4: delta=0.656100
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  0.0 
   2 0.66 0.73 0.81 0.73 
-------------------------
iter 5: delta=0.000000
  r/c   0    1    2    3 
   0 0.

In [256]:
def policy_iteration(
    env: GridWorld,
    gamma=0.9,
    theta=1e-6,
    max_iters=1000,
    verbose=0,
    pe_max_iters=1000,
    pe_verbose=0,
    policy: np.ndarray = None
):
    if policy is None:
        policy = np.zeros((env.nS, env.nA))
        policy[np.arange(env.nS)] = np.array([1.0, 0, 0, 0])
    V = np.zeros(env.nS)

    if verbose >= 2:
        print('---------- policy_iteration ------------')
        print('policy init')
        render_policy(env, policy)

    i = 0
    pe_converged = True
    changed = None
    while changed != 0 and i < max_iters:
        V, pe_converged = policy_evaluation(
            env, policy, gamma, theta, max_iters=pe_max_iters, verbose=pe_verbose
        )
        if not pe_converged:
            break
        new_policy = read_policy(env, V, gamma)
        changed = int(
            np.count_nonzero(np.argmax(new_policy, axis=1) != np.argmax(policy, axis=1))
        )
        if verbose >= 2:
            print('-' * 25)
            print(f'iter {i}: V')
            show_V(env, V)
            print('-' * 5)  
        if verbose >= 1:
            prefix = f'iter {i}: ' if verbose == 1 else ''
            print(f'{prefix}actions changed = {changed}')
        if verbose >= 2:
            render_policy(env, new_policy)
        policy = new_policy
        i += 1
    converged = changed == 0
    if verbose >= 1:
        print('-' * 25)  
        if not pe_converged:
            print(
                f"policy_iteration did not converge because policy_evaluation did not converge"
            )
        elif converged:
            print(f'policy_iteration converged in {i - 1} iterations')
        else:
            print(f'policy_iteration did not converge in {max_iters} iterations')
    return policy, V, converged    

In [257]:
policy, V, converged = policy_iteration(env, verbose=2)

---------- policy_iteration ------------
policy init
 r/c  0   1   2   3 
  0   ↑   ↑   ↑  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ↑ 
-------------------------
iter 0: V
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0    #  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-----
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ← 
-------------------------
iter 1: V
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0    #  0.9  0.0 
   2  0.0  0.0 0.81 0.73 
-----
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 
-------------------------
iter 2: V
  r/c   0    1    2    3 
   0  0.0  0.9  1.0  0.0 
   1  0.0    #  0.9  0.0 
   2  0.0 0.73 0.81 0.73 
-----
actions changed = 2
 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   →   →   ↑   ← 
-------------------------
iter 3: V
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  0.0 

### uniform policy

In [258]:
uniform_policy = np.ones((env.nS, env.nA)) / env.nA
policy, V, converged = policy_iteration(env, policy=uniform_policy, verbose=2)

---------- policy_iteration ------------
policy init
 r/c  0   1   2   3 
  0   ↑   ↑   ↑  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ↑ 
-------------------------
iter 0: V
  r/c   0    1    2    3 
   0 0.05 0.13 0.26  0.0 
   1 -0.01    # -0.34  0.0 
   2 -0.07 -0.15 -0.31 -0.58 
-----
actions changed = 6
 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ←   ←   ← 
-------------------------
iter 1: V
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  0.0 
   2 0.66 0.59 0.53 0.48 
-----
actions changed = 1
 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ←   ↑   ← 
-------------------------
iter 2: V
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  0.0 
   2 0.66 0.59 0.81 0.73 
-----
actions changed = 1
 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 
-------------------------
iter 3: V
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9

### analytic_policy_value

In [259]:
def analytic_policy_value(env: GridWorld, policy: np.ndarray, gamma=0.9):
    n = env.nS
    Ppi = np.zeros((n, n))
    rpi = np.zeros(n)
    for s in range(n):
        if env.s2c[s] in env.terminals:
            continue
        for a in range(env.nA):
            pa = policy[s][a]
            ns, r = env.step(s, a)
            prob = 1.0
            Ppi[s, ns] += pa * prob
            rpi[s] += pa * prob * r
    return np.linalg.solve(np.eye(n) - gamma * Ppi, rpi)

In [260]:
# run_if: cell_allowed()
policy, V_star, converged = value_iteration(env)
V = analytic_policy_value(env, policy, 0.9)
show_V(env, V)
print('-' * 25)
equal = np.array_equal(V, V_star)
print(f'V derived by analytic_policy_value equal to V_star from value_iteration = {equal}')

  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73    #  0.9  0.0 
   2 0.66 0.73 0.81 0.73 
-------------------------
V derived by analytic_policy_value equal to V_star from value_iteration = True
